# One-file minimal sanity validation

这个 notebook 是前面 4 个 sanity experiments 的合并精简版。

目标：**只跑一个 notebook，只输出一份最终报告**：

```text
MMRL/output_refactor/analysis/sanity_onefile_report/.../sanity_onefile_report.json
```

它默认跑 3 个轻量实验：

1. text-support conflict 是否预测 final error
2. support label noise 下 conflict 是否比 entropy 更敏感
3. prompt mismatch 下 conflict 是否识别 text prior 不可靠

第 4 个 OOD / near-OOD 实验默认不跑，因为 ImageNet-R/A/Sketch 较重。需要时打开 `RUN_OOD = True`。

最终报告里会统一给出：

- 每个 sanity experiment 的关键指标
- 是否通过
- kill / pause / continue 建议


In [ ]:
from __future__ import annotations

import os
import sys
import json
import math
import importlib
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")


## 1. 只改这里

把 notebook 放在 `MMRL/notebooks/` 下运行。  
默认从已有 adapter checkpoint 加载模型，但 sanity test 的 support evidence 会重新构造。


In [ ]:
# ---- project ----
PROJECT_ROOT = Path("..").resolve()
assert (PROJECT_ROOT / "run.py").exists(), f"PROJECT_ROOT seems wrong: {PROJECT_ROOT}"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT = "DATASETS"
OUTPUT_ROOT = PROJECT_ROOT / "output_refactor"

# ---- experiment config ----
METHOD_ALIAS = "CLAP"          # "CLAP", "TipA", "BayesAdapter", "PP_PROKER_ONEHOT"
DATASET = "caltech101"
SHOTS = 16
SEED = 1
BACKBONE = "ViT-B/16"

# ---- what to run ----
RUN_LABEL_NOISE = True
RUN_PROMPT_MISMATCH = True
RUN_OOD = False                # heavy; turn on only after ImageNet-R/A/Sketch are ready

LABEL_NOISE_RATES = [0.0, 0.10, 0.20, 0.40]
PROMPT_MODES = ["original", "weak_blend_mean", "coarse_groups", "wrong_permutation"]

# ---- support-cache knobs ----
CACHE_BETA = 20.0
CACHE_TOPK = 64
FINAL_BLEND = 0.5
CONFLICT_GAMMA = 5.0

# ---- decision thresholds ----
MIN_AUROC_GAIN = 0.01
MIN_JS_AUROC_FOR_PROMPT_ERROR = 0.80

RUN_TAG_BY_ALIAS = {
    "ZS": "ZS",
    "CLAP": "CLAP",
    "PP_PROKER_ONEHOT": "PP_PROKER_ONEHOT",
    "ECKA": "ECKA",
    "CAPEL": "CAPEL",
    "VNC_CAPEL": "VNC_CAPEL",
    "BayesAdapter": "BAYES_ADAPTER",
    "BAYES_ADAPTER": "BAYES_ADAPTER",
    "TipA": "TipA",
    "TipA-f-": "TipA-f-",
    "CrossModal": "CrossModal",
}

RUN_TAG = RUN_TAG_BY_ALIAS.get(METHOD_ALIAS, METHOD_ALIAS)
BACKBONE_DIR = BACKBONE.replace("/", "-")

MODEL_DIR = OUTPUT_ROOT / "ClipAdapters" / RUN_TAG / "FS" / "fewshot_train" / DATASET / f"shots_{SHOTS}" / BACKBONE_DIR / f"seed{SEED}"
OUT_DIR = OUTPUT_ROOT / "analysis" / "sanity_onefile_report" / RUN_TAG / DATASET / f"shots_{SHOTS}" / BACKBONE_DIR / f"seed{SEED}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = OUT_DIR / "sanity_onefile_report.json"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_DIR:", MODEL_DIR, "exists:", MODEL_DIR.exists())
print("REPORT_PATH:", REPORT_PATH)


## 2. 一键运行

下面这个 cell 会完成所有轻量 sanity experiments，并只保存一份 JSON 报告。


In [ ]:
# -----------------------------
# Config mapping / trainer build
# -----------------------------

METHOD_CONFIG_BY_ALIAS = {
    "ZS": "configs/methods/clip_adapters_zs.yaml",
    "CLAP": "configs/methods/clip_adapters_clap.yaml",
    "PP_PROKER_ONEHOT": "configs/methods/clip_adapters_pp_proker_onehot.yaml",
    "ECKA": "configs/methods/clip_adapters_ecka.yaml",
    "CAPEL": "configs/methods/clip_adapters_capel.yaml",
    "VNC_CAPEL": "configs/methods/clip_adapters_vnccapel.yaml",
    "RANDOM": "configs/methods/clip_adapters_random.yaml",
    "TR": "configs/methods/clip_adapters_tr.yaml",
    "TaskRes": "configs/methods/clip_adapters_tr.yaml",
    "ClipA": "configs/methods/clip_adapters_clipa.yaml",
    "CLIPA": "configs/methods/clip_adapters_clipa.yaml",
    "TipA": "configs/methods/clip_adapters_tipa.yaml",
    "TipA-f-": "configs/methods/clip_adapters_tipa_f.yaml",
    "TipA-F": "configs/methods/clip_adapters_tipa_f.yaml",
    "CrossModal": "configs/methods/clip_adapters_crossmodal.yaml",
    "BayesAdapter": "configs/methods/clip_adapters_bayes.yaml",
    "BAYES_ADAPTER": "configs/methods/clip_adapters_bayes.yaml",
    "BayesAdapter_l2": "configs/methods/clip_adapters_bayes_clap.yaml",
    "BAYES_ADAPTER_l2": "configs/methods/clip_adapters_bayes_clap.yaml",
}

def register_project():
    from core.utils import import_optional_modules
    import_optional_modules([
        "datasets.oxford_pets", "datasets.oxford_flowers", "datasets.fgvc_aircraft",
        "datasets.dtd", "datasets.eurosat", "datasets.stanford_cars", "datasets.food101",
        "datasets.sun397", "datasets.caltech101", "datasets.ucf101", "datasets.imagenet",
        "datasets.imagenetv2", "datasets.imagenet_sketch", "datasets.imagenet_a", "datasets.imagenet_r",
    ])
    importlib.import_module("trainers.refactor_runner")

def build_trainer_onefile(method_alias, dataset, shots, seed, out_dir, load_model=True):
    from dassl.engine import build_trainer
    from dassl.utils import set_random_seed, setup_logger
    from core.config import setup_cfg

    register_project()

    if seed >= 0:
        set_random_seed(seed)

    method_cfg = METHOD_CONFIG_BY_ALIAS[method_alias]
    dataset_cfg = f"configs/datasets/{dataset}.yaml"
    protocol_cfg = "configs/protocols/fs.yaml"
    runtime_cfg = "configs/runtime/adapter_family.yaml"

    for rel in [method_cfg, dataset_cfg, protocol_cfg, runtime_cfg]:
        assert (PROJECT_ROOT / rel).exists(), f"Missing config: {PROJECT_ROOT / rel}"

    args = SimpleNamespace(
        root=str(PROJECT_ROOT / DATA_ROOT),
        output_dir=str(out_dir),
        dataset_config_file=str(PROJECT_ROOT / dataset_cfg),
        method_config_file=str(PROJECT_ROOT / method_cfg),
        protocol_config_file=str(PROJECT_ROOT / protocol_cfg),
        runtime_config_file=str(PROJECT_ROOT / runtime_cfg),
        exp_config="",
        method="ClipAdapters",
        protocol="FS",
        exec_mode="cache",
        seed=seed,
        trainer="RefactorRunner",
        eval_only=True,
        model_dir=str(MODEL_DIR),
        load_epoch=None,
        no_train=True,
        opts=[
            "DATASET.NUM_SHOTS", str(shots),
            "DATASET.SUBSAMPLE_CLASSES", "all",
            "MODEL.BACKBONE.NAME", BACKBONE,
        ],
    )

    cfg = setup_cfg(args)
    setup_logger(cfg.OUTPUT_DIR)
    trainer = build_trainer(cfg)

    if load_model and MODEL_DIR.exists():
        trainer.load_model(str(MODEL_DIR), epoch=None)
    elif load_model:
        print("[WARN] MODEL_DIR not found; using freshly built model:", MODEL_DIR)

    trainer.set_model_mode("eval")
    return trainer

# -----------------------------
# Math helpers
# -----------------------------

EPS = 1e-12

def softmax_np(logits):
    logits = np.asarray(logits, dtype=np.float64)
    z = logits - np.max(logits, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.clip(e.sum(axis=1, keepdims=True), EPS, None)

def norm_probs(p):
    p = np.clip(np.asarray(p, dtype=np.float64), EPS, None)
    return p / np.clip(p.sum(axis=1, keepdims=True), EPS, None)

def entropy_np(p):
    p = norm_probs(p)
    return -(p * np.log(p)).sum(axis=1)

def js_np(p, q):
    p = norm_probs(p)
    q = norm_probs(q)
    m = 0.5 * (p + q)
    return 0.5 * ((p * (np.log(p) - np.log(m))).sum(axis=1) + (q * (np.log(q) - np.log(m))).sum(axis=1))

def margin_np(p):
    p = norm_probs(p)
    top2 = np.partition(p, kth=-2, axis=1)[:, -2:]
    top2.sort(axis=1)
    return top2[:, 1] - top2[:, 0]

def energy_np(logits):
    return (-torch.logsumexp(torch.tensor(logits).float(), dim=1)).numpy()

def safe_auc(y, s):
    y = np.asarray(y).astype(int)
    s = np.asarray(s, dtype=float)
    mask = np.isfinite(s)
    if mask.sum() < 2 or len(np.unique(y[mask])) < 2:
        return float("nan")
    return float(roc_auc_score(y[mask], s[mask]))

def aurc(error, score):
    error = np.asarray(error).astype(float)
    score = np.asarray(score, dtype=float)
    mask = np.isfinite(score)
    error = error[mask]
    score = score[mask]
    if len(error) == 0:
        return float("nan")
    order = np.argsort(score)
    e = error[order]
    risks = np.cumsum(e) / np.arange(1, len(e) + 1)
    return float(np.mean(risks))

def optimal_aurc(error):
    e = np.asarray(error).astype(float)
    order = np.argsort(e)
    e = e[order]
    risks = np.cumsum(e) / np.arange(1, len(e) + 1)
    return float(np.mean(risks))

def det_metrics(df, error_col, score_cols):
    rows = []
    e = df[error_col].to_numpy().astype(int)
    opt = optimal_aurc(e)
    for c in score_cols:
        a = safe_auc(e, df[c].to_numpy())
        r = aurc(e, df[c].to_numpy())
        rows.append({"score": c, "auroc": a, "aurc": r, "eaurc": r - opt})
    return pd.DataFrame(rows).sort_values("auroc", ascending=False)

# -----------------------------
# Feature / prediction helpers
# -----------------------------

@torch.no_grad()
def collect_features(trainer, loader=None):
    if loader is None:
        loader = trainer.test_loader

    model = trainer.method.model
    device = trainer.device
    feats, labels = [], []

    for batch in loader:
        image = batch["img"].to(device)
        label = batch["label"].to(device).long()
        try:
            f = model.image_encoder(image.type(model.dtype))
        except Exception:
            f = model.image_encoder(image.float())
        feats.append(f.detach().cpu().float())
        labels.append(label.detach().cpu().long())

    return torch.cat(feats, dim=0), torch.cat(labels, dim=0)

def support_features(trainer):
    if hasattr(trainer, "features_train") and hasattr(trainer, "labels_train"):
        return trainer.features_train.detach().cpu().float(), trainer.labels_train.detach().cpu().long()
    return collect_features(trainer, trainer.train_loader_x)

@torch.no_grad()
def text_probs(trainer, query_features, text_features_override=None):
    model = trainer.method.model
    device = trainer.device

    if text_features_override is None:
        text_f = model.adapter.base_text_features.detach().cpu().float()
    else:
        text_f = text_features_override.detach().cpu().float()

    q = F.normalize(query_features.float().to(device), dim=-1)
    t = F.normalize(text_f.float().to(device), dim=-1)
    logits = q @ t.t() * model.logit_scale.exp().detach().to(device)
    logits = logits.detach().cpu().numpy()
    return softmax_np(logits), logits

def cache_probs(query_features, supp_features, supp_labels, num_classes, beta=CACHE_BETA):
    q = F.normalize(query_features.float(), dim=-1)
    s = F.normalize(supp_features.float(), dim=-1)
    onehot = F.one_hot(supp_labels.long(), num_classes=num_classes).float()

    probs, logits, density = [], [], []
    bs = 1024

    for i in range(0, q.shape[0], bs):
        sim = q[i:i+bs] @ s.t()
        aff = torch.softmax(beta * sim, dim=1)
        p = aff @ onehot
        p = p / p.sum(dim=1, keepdim=True).clamp_min(EPS)

        k = min(CACHE_TOPK, sim.shape[1])
        topv = torch.topk(sim, k=k, dim=1).values

        probs.append(p.cpu())
        logits.append(torch.log(p.clamp_min(EPS)).cpu())
        density.append(torch.stack([sim.max(dim=1).values, topv.mean(dim=1)], dim=1).cpu())

    return (
        torch.cat(probs, dim=0).numpy(),
        torch.cat(logits, dim=0).numpy(),
        torch.cat(density, dim=0).numpy(),
    )

def build_df(trainer, supp_f, supp_y, query_f=None, query_y=None, text_features_override=None):
    names = list(trainer.dm.dataset.classnames)
    C = len(names)

    if query_f is None or query_y is None:
        query_f, query_y = collect_features(trainer)

    p_text, text_logits = text_probs(trainer, query_f, text_features_override)
    p_support, support_logits, density = cache_probs(query_f, supp_f, supp_y, C)
    p_final = norm_probs((1.0 - FINAL_BLEND) * p_text + FINAL_BLEND * p_support)

    y = query_y.numpy().astype(int)
    pred_text = p_text.argmax(axis=1)
    pred_support = p_support.argmax(axis=1)
    pred_final = p_final.argmax(axis=1)

    df = pd.DataFrame({
        "y": y,
        "pred_text": pred_text,
        "pred_support": pred_support,
        "pred_final": pred_final,
        "correct_text": (pred_text == y).astype(int),
        "correct_support": (pred_support == y).astype(int),
        "correct_final": (pred_final == y).astype(int),
    })

    for src in ["text", "support", "final"]:
        df[f"error_{src}"] = 1 - df[f"correct_{src}"]

    df["js"] = js_np(p_text, p_support)
    df["top1_disagree"] = (pred_text != pred_support).astype(int)
    df["margin_conflict"] = df["top1_disagree"] * (
        np.maximum(p_text.max(axis=1) - p_text[np.arange(len(y)), pred_support], 0.0)
        + np.maximum(p_support.max(axis=1) - p_support[np.arange(len(y)), pred_text], 0.0)
    )

    df["msp_text"] = p_text.max(axis=1)
    df["msp_support"] = p_support.max(axis=1)
    df["msp_final"] = p_final.max(axis=1)

    df["entropy_text"] = entropy_np(p_text)
    df["entropy_support"] = entropy_np(p_support)
    df["entropy_final"] = entropy_np(p_final)

    df["margin_text"] = margin_np(p_text)
    df["margin_support"] = margin_np(p_support)
    df["margin_final"] = margin_np(p_final)

    df["energy_text"] = energy_np(text_logits)
    df["energy_support"] = energy_np(support_logits)
    df["energy_final"] = energy_np(np.log(np.clip(p_final, EPS, 1.0)))

    df["one_minus_msp_final"] = 1.0 - df["msp_final"]
    df["neg_margin_final"] = -df["margin_final"]
    df["conflict_aware_error_score"] = 1.0 - (df["msp_final"] / (1.0 + CONFLICT_GAMMA * df["js"]))

    df["support_density_max"] = density[:, 0]
    df["support_density_topk"] = density[:, 1]
    df["ignorance_score"] = -df["support_density_topk"]

    return df

def corrupt_labels(labels, num_classes, noise_rate, seed):
    rng = np.random.default_rng(seed)
    y = labels.detach().cpu().numpy().copy()
    m = int(round(noise_rate * len(y)))
    if m <= 0:
        return torch.tensor(y, dtype=torch.long)
    idx = rng.choice(len(y), size=m, replace=False)
    for i in idx:
        old = int(y[i])
        new = int(rng.integers(0, num_classes - 1))
        if new >= old:
            new += 1
        y[i] = new
    return torch.tensor(y, dtype=torch.long)

def prompt_features(trainer, mode, seed=0):
    base = trainer.method.model.adapter.base_text_features.detach().cpu().float()
    C, D = base.shape
    rng = np.random.default_rng(seed)

    if mode == "original":
        return base.clone()

    if mode == "weak_blend_mean":
        mean = base.mean(dim=0, keepdim=True)
        return F.normalize(0.35 * base + 0.65 * mean, dim=-1)

    if mode == "wrong_permutation":
        perm = rng.permutation(C)
        if C > 1:
            for i in range(C):
                if perm[i] == i:
                    j = (i + 1) % C
                    perm[i], perm[j] = perm[j], perm[i]
        return base[torch.tensor(perm, dtype=torch.long)]

    if mode == "coarse_groups":
        n_groups = max(2, int(round(math.sqrt(C))))
        order = np.arange(C)
        groups = np.array_split(order, n_groups)
        out = base.clone()
        for g in groups:
            mean = base[torch.tensor(g, dtype=torch.long)].mean(dim=0, keepdim=True)
            out[torch.tensor(g, dtype=torch.long)] = mean.expand(len(g), -1)
        return F.normalize(out, dim=-1)

    raise ValueError(mode)

def logreg_delta(df):
    base_feats = ["entropy_final", "margin_final", "msp_final"]
    plus_feats = base_feats + ["js"]

    def fit(feats):
        sub = df[["error_final"] + feats].replace([np.inf, -np.inf], np.nan).dropna()
        y = sub["error_final"].astype(int).to_numpy()
        X = sub[feats].to_numpy(float)
        if len(np.unique(y)) < 2:
            return {"auroc": np.nan, "eaurc": np.nan}
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
        clf.fit(Xtr, ytr)
        s = clf.predict_proba(Xte)[:, 1]
        r = aurc(yte, s)
        coefs = dict(zip(feats, clf.named_steps["logisticregression"].coef_[0]))
        return {"auroc": safe_auc(yte, s), "aurc": r, "eaurc": r - optimal_aurc(yte), "coefs": coefs}

    b = fit(base_feats)
    p = fit(plus_feats)
    return {
        "confidence_only": b,
        "confidence_plus_js": p,
        "delta_auroc": p["auroc"] - b["auroc"],
        "delta_eaurc": p["eaurc"] - b["eaurc"],
        "js_coef": p.get("coefs", {}).get("js", None),
    }

# -----------------------------
# Run
# -----------------------------

trainer = build_trainer_onefile(METHOD_ALIAS, DATASET, SHOTS, SEED, OUT_DIR, load_model=True)
names = list(trainer.dm.dataset.classnames)
num_classes = len(names)

supp_f, supp_y = support_features(trainer)
query_f, query_y = collect_features(trainer)

report = {
    "config": {
        "method_alias": METHOD_ALIAS,
        "run_tag": RUN_TAG,
        "dataset": DATASET,
        "shots": SHOTS,
        "seed": SEED,
        "backbone": BACKBONE,
        "model_dir": str(MODEL_DIR),
        "n_support": int(len(supp_y)),
        "n_query": int(len(query_y)),
        "num_classes": int(num_classes),
    },
    "experiments": {},
    "decision": {},
}

# Experiment 1
df1 = build_df(trainer, supp_f, supp_y, query_f, query_y)
scores = ["js", "top1_disagree", "margin_conflict", "one_minus_msp_final", "entropy_final", "neg_margin_final", "energy_final", "conflict_aware_error_score"]
m1 = det_metrics(df1, "error_final", scores)
lr1 = logreg_delta(df1)

report["experiments"]["exp1_text_support_conflict"] = {
    "final_accuracy": float(df1["correct_final"].mean()),
    "text_accuracy": float(df1["correct_text"].mean()),
    "support_accuracy": float(df1["correct_support"].mean()),
    "text_support_disagreement": float(df1["top1_disagree"].mean()),
    "metrics": m1.to_dict(orient="records"),
    "controlled_logreg": lr1,
}

# Experiment 2
if RUN_LABEL_NOISE:
    noise_rows = []
    for noise in LABEL_NOISE_RATES:
        noisy_y = corrupt_labels(supp_y, num_classes, noise, SEED + int(noise * 1000))
        dfn = build_df(trainer, supp_f, noisy_y, query_f, query_y)
        mn = det_metrics(dfn, "error_final", scores)

        def get(metric_name, col):
            x = mn.loc[mn.score == metric_name, col]
            return float(x.iloc[0]) if len(x) else np.nan

        noise_rows.append({
            "noise_rate": float(noise),
            "final_accuracy": float(dfn["correct_final"].mean()),
            "support_accuracy": float(dfn["correct_support"].mean()),
            "mean_js": float(dfn["js"].mean()),
            "mean_entropy_final": float(dfn["entropy_final"].mean()),
            "auroc_js": get("js", "auroc"),
            "auroc_entropy_final": get("entropy_final", "auroc"),
            "auroc_one_minus_msp_final": get("one_minus_msp_final", "auroc"),
            "aurc_conflict_aware": get("conflict_aware_error_score", "aurc"),
            "aurc_one_minus_msp_final": get("one_minus_msp_final", "aurc"),
        })
    report["experiments"]["exp2_support_label_noise"] = noise_rows

# Experiment 3
if RUN_PROMPT_MISMATCH:
    prompt_rows = []
    for mode in PROMPT_MODES:
        tf = prompt_features(trainer, mode, seed=SEED)
        dfp = build_df(trainer, supp_f, supp_y, query_f, query_y, text_features_override=tf)
        mp_text = det_metrics(dfp, "error_text", ["js", "top1_disagree", "margin_conflict", "entropy_text", "one_minus_msp_final", "ignorance_score"])
        mp_final = det_metrics(dfp, "error_final", scores)

        def get(dfm, metric_name, col):
            x = dfm.loc[dfm.score == metric_name, col]
            return float(x.iloc[0]) if len(x) else np.nan

        prompt_rows.append({
            "prompt_mode": mode,
            "text_accuracy": float(dfp["correct_text"].mean()),
            "support_accuracy": float(dfp["correct_support"].mean()),
            "final_accuracy": float(dfp["correct_final"].mean()),
            "mean_js": float(dfp["js"].mean()),
            "auroc_js_for_text_error": get(mp_text, "js", "auroc"),
            "auroc_entropy_text_for_text_error": get(mp_text, "entropy_text", "auroc"),
            "auroc_js_for_final_error": get(mp_final, "js", "auroc"),
            "auroc_conflict_aware_for_final_error": get(mp_final, "conflict_aware_error_score", "auroc"),
        })
    report["experiments"]["exp3_prompt_mismatch"] = prompt_rows

# Optional OOD placeholder
if RUN_OOD:
    report["experiments"]["exp4_ood"] = {
        "status": "not_implemented_in_onefile_light_version",
        "note": "Use the heavier notebook only if ImageNet-R/A/Sketch dataloaders are ready."
    }

# Decision
reasons = []
passed = []

# Exp1 decision
e1 = report["experiments"]["exp1_text_support_conflict"]
m1df = pd.DataFrame(e1["metrics"])
def metric_auroc(score):
    x = m1df.loc[m1df.score == score, "auroc"]
    return float(x.iloc[0]) if len(x) else np.nan

auroc_js = metric_auroc("js")
auroc_entropy = metric_auroc("entropy_final")
auroc_msp = metric_auroc("one_minus_msp_final")
delta_auc = e1["controlled_logreg"]["delta_auroc"]
delta_eaurc = e1["controlled_logreg"]["delta_eaurc"]

if np.isfinite(auroc_js) and auroc_js > max(auroc_entropy, auroc_msp):
    passed.append("Exp1: JS beats entropy/MSP as an error detector.")
else:
    reasons.append("Exp1 fail: JS does not beat entropy/MSP error-detection baselines.")

if np.isfinite(delta_auc) and delta_auc >= MIN_AUROC_GAIN:
    passed.append("Exp1 controlled: adding JS improves AUROC by >= threshold.")
else:
    reasons.append(f"Exp1 controlled fail: adding JS improves AUROC by < {MIN_AUROC_GAIN}.")

if np.isfinite(delta_eaurc) and delta_eaurc < 0:
    passed.append("Exp1 controlled: adding JS decreases E-AURC.")
else:
    reasons.append("Exp1 controlled fail: adding JS does not decrease E-AURC.")

# Exp2 decision
if RUN_LABEL_NOISE:
    e2 = pd.DataFrame(report["experiments"]["exp2_support_label_noise"])
    high = e2.sort_values("noise_rate").tail(1).iloc[0]
    if high["auroc_js"] > high["auroc_entropy_final"]:
        passed.append("Exp2: at highest label noise, JS is more sensitive than entropy.")
    else:
        reasons.append("Exp2 fail: at highest label noise, JS is not more sensitive than entropy.")

    if high["aurc_conflict_aware"] < high["aurc_one_minus_msp_final"]:
        passed.append("Exp2: conflict-aware selective risk proxy improves AURC.")
    else:
        reasons.append("Exp2 fail: conflict-aware selective risk proxy does not improve AURC.")

# Exp3 decision
if RUN_PROMPT_MISMATCH:
    e3 = pd.DataFrame(report["experiments"]["exp3_prompt_mismatch"])
    wrong = e3[e3.prompt_mode == "wrong_permutation"]
    if len(wrong) and float(wrong.iloc[0]["auroc_js_for_text_error"]) >= MIN_JS_AUROC_FOR_PROMPT_ERROR:
        passed.append("Exp3: JS identifies wrong text prior under wrong_permutation.")
    else:
        reasons.append("Exp3 fail: JS does not strongly identify wrong text prior.")

if len(reasons) == 0:
    recommendation = "CONTINUE"
elif len(passed) >= 2:
    recommendation = "PAUSE_AND_TEST_SHIFT_OR_NOISE"
else:
    recommendation = "KILL_OR_PAUSE"

report["decision"] = {
    "recommendation": recommendation,
    "passed_checks": passed,
    "failed_or_missing_checks": reasons,
}

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(json.dumps(report["decision"], indent=2, ensure_ascii=False))
print("saved one report:", REPORT_PATH)

print("\nExp1 metrics:")
display(m1)

if RUN_LABEL_NOISE:
    print("\nExp2 label noise summary:")
    display(pd.DataFrame(report["experiments"]["exp2_support_label_noise"]))

if RUN_PROMPT_MISMATCH:
    print("\nExp3 prompt mismatch summary:")
    display(pd.DataFrame(report["experiments"]["exp3_prompt_mismatch"]))


## 3. 结果怎么看

只需要看最后打印的 `decision` 和唯一输出文件：

```text
sanity_onefile_report.json
```

判断规则：

- `CONTINUE`: 轻量 sanity 基本过关，值得继续做更重的 shift/OOD/noise 实验。
- `PAUSE_AND_TEST_SHIFT_OR_NOISE`: 部分信号存在，但还不够；优先跑 label noise 或 ImageNet-R/A/Sketch。
- `KILL_OR_PAUSE`: 当前切口不支持继续做复杂 fusion。

重点字段：

```text
experiments.exp1_text_support_conflict.controlled_logreg.delta_auroc
experiments.exp1_text_support_conflict.controlled_logreg.delta_eaurc
experiments.exp2_support_label_noise[-1].auroc_js
experiments.exp2_support_label_noise[-1].auroc_entropy_final
experiments.exp3_prompt_mismatch[wrong_permutation].auroc_js_for_text_error
```
